### Zweck des Validierungs-Notebooks

Dieses Notebook dient der **fachlichen Validierung der elektrischen Berechnungslogik** (DC-Loadflow und PTDF-Ableitung), die im Produktivskript `battery_bands.py` verwendet wird.

Anhand eines **kleinen, künstlich aufgebauten Testnetzes** (JSON-Graph mit wenigen Knoten und Leitungen) wird gezeigt, dass:

1. aus den in der JSON hinterlegten **Leitungsreaktanzen** korrekt eine Suszeptanzmatrix (B-Matrix) aufgebaut wird,
2. daraus **Power Transfer Distribution Factors (PTDF)** gemäß der im Produktivcode verwendeten Formel berechnet werden,
3. die mit PTDF berechneten Leitungsänderungen **identisch** sind zu den Leitungsänderungen eines explizit gelösten **DC-Lastflusses** (Finite-Difference-Vergleich),
4. die aus Stromgrenzen abgeleiteten **Leistungsgrenzen und Auslastungen** konsistent interpretiert werden.

Das Notebook repliziert dabei **bewusst keine eigene Logik**, sondern verwendet – soweit möglich – dieselben Hilfsfunktionen und Konventionen wie `battery_bands.py`.  
Es beantwortet ausschließlich die Frage:

> *„Ist die mathematische Herleitung und Implementierung der elektrischen Netzberechnung korrekt?“*

Nicht Gegenstand dieses Notebooks sind:
- Forecast-Qualität,
- zeitliche Simulation über viele Zeitpunkte,
- Optimierung oder Ableitung von Leistungsbändern.

Damit fungiert das Notebook als **nachvollziehbarer Rechen- und Plausibilitätsnachweis** für die in der Masterarbeit beschriebene Netzmodellierung.


### Imports + Config

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from src import config as cfg
from src.battery_bands import (
    edge_raw_X_ohm,
    ohm_to_pu,
    amp_to_mw_limit,
    connected_components,
    choose_slack_for_component,
)

GRAPH_PATH = Path("src/data/raw/graph/validate_graph.json")

SLACK_OVERRIDE = "N01"

# Konfig wie in battery_bands.py
S_BASE_MVA      = float(cfg.S_BASE_MVA)
V_KV_DEFAULT    = float(cfg.V_KV_DEFAULT)
COSPHI_MIN      = float(cfg.COSPHI_MIN)
UTIL_TARGET_PCT = float(cfg.UTIL_TARGET_PCT)
X_EPS_OHM       = float(cfg.X_EPS_OHM)

util_scale = UTIL_TARGET_PCT / 100.0
assert 0.0 < util_scale <= 1.0, "UTIL_TARGET_PCT muss in (0,100] liegen."


### Graph laden + Nodes/Edges extrahieren

In [13]:
graph = json.loads(GRAPH_PATH.read_text(encoding="utf-8"))

nodes, edges = [], []
node_types, node_features = {}, {}

for it in graph:
    if "data" not in it:
        continue
    d = it["data"]
    if "source" in d and "target" in d:
        edges.append(d)
    else:
        nid = d["id"]
        nodes.append(nid)
        node_types[nid] = (d.get("type") or "").strip()
        node_features[nid] = d.get("features", {}) or {}

print(f"Graph: nodes={len(nodes)} edges={len(edges)}")
print("Node types:", pd.Series(list(node_types.values())).value_counts().to_dict())

assert SLACK_OVERRIDE in nodes, f"Slack {SLACK_OVERRIDE} nicht in nodes."


Graph: nodes=4 edges=3
Node types: {'uw_field': 2, 'battery': 1, 'junction': 1}


### Validierungs-Setup: Keine Kontraktion (jeder Node ist Supernode)
-> wir nehmen nur Kanten mit X_total_ohm > X_EPS_OHM

In [14]:
super_nodes = nodes[:]          
rep_map = {n: n for n in nodes} 

lines_ohm = []
edge_meta = {}  

for e in edges:
    u, v = e["source"], e["target"]
    feat = e.get("features", {}) or {}
    eid = e.get("id") or feat.get("id") or f"{u}-{v}"

    Xohm = edge_raw_X_ohm(feat)
    if Xohm is None:
        raise ValueError(f"Edge {eid}: keine X_total_ohm/X_Total_Ohm gefunden. feat={feat}")

    Xohm = float(Xohm)
    if Xohm <= float(X_EPS_OHM):
        # würde in run() kontrahiert werden -> wird für die Validierung ignoriert
        continue

    ru, rv = rep_map[u], rep_map[v]
    if ru == rv:
        continue

    lines_ohm.append((ru, rv, eid, feat, Xohm))
    edge_meta[eid] = {
        "u": ru,
        "v": rv,
        "Xohm": Xohm,
        "limit_A": feat.get("Strom_Limit_in_A", None),
    }

print(f"Kept edges (no contraction): {len(lines_ohm)}")

df_edges = pd.DataFrame([
    {
        "edge_id": eid,
        "u": edge_meta[eid]["u"],
        "v": edge_meta[eid]["v"],
        "X_total_ohm": edge_meta[eid]["Xohm"],
        "limit_A": edge_meta[eid]["limit_A"],
        "P_limit_MW@UTIL": (
            amp_to_mw_limit(float(edge_meta[eid]["limit_A"]), V_KV_DEFAULT, COSPHI_MIN) * util_scale
            if edge_meta[eid]["limit_A"] not in (None, "") else np.nan
        ),
    }
    for (_, _, eid, _, _) in lines_ohm
])

df_edges


Kept edges (no contraction): 3


,edge_id,u,v,X_total_ohm,limit_A,P_limit_MW@UTIL
0,E01,N01,J01,0.02,900,81.449689
1,E02,J01,N02,0.02,600,54.299793
2,E03,J01,BESS01,0.10,400,36.199862


### B-Matrix (p.u.) + lines_pu

In [15]:
node_index = {nid: i for i, nid in enumerate(super_nodes)}
n = len(super_nodes)
B = np.zeros((n, n), dtype=float)

# lines_pu: (u, v, Xpu, eid, feat)
lines_pu = []
for (u, v, eid, feat, Xohm) in lines_ohm:
    Xpu = ohm_to_pu(Xohm, V_KV_DEFAULT, S_BASE_MVA)
    if Xpu <= 0:
        continue
    lines_pu.append((u, v, float(Xpu), eid, feat))

assert len(lines_pu) > 0, "Keine gültigen Leitungen (Xpu>0)."

# B off-diagonal
for (u, v, Xpu, eid, feat) in lines_pu:
    i, j = node_index[u], node_index[v]
    b = 1.0 / Xpu
    B[i, j] -= b
    B[j, i] -= b

# B diagonal
for i in range(n):
    B[i, i] = -np.sum(B[i, :])

print("B shape:", B.shape)


B shape: (4, 4)


### Komponenten + Slack (fixer Slack Override)

In [16]:
components = connected_components(super_nodes, [(u, v, Xpu, eid, feat) for (u, v, Xpu, eid, feat) in lines_pu])

slack_by_comp = {
    frozenset(comp): choose_slack_for_component(comp, [(u, v, Xpu, eid, feat) for (u, v, Xpu, eid, feat) in lines_pu], fixed_slack=SLACK_OVERRIDE)
    for comp in components
}

print("Components:", components)
print("Slack by comp:", {tuple(sorted(list(k))): v for k, v in slack_by_comp.items()})


Components: [['N01', 'J01', 'N02', 'BESS01']]
Slack by comp: {('BESS01', 'J01', 'N01', 'N02'): 'N01'}


### PTDF je Komponente (wie in battery_bands.run())

-> hier bauen wir PTDF explizit, weil run() nichts zurückgibt.

In [17]:
PTDF_by_comp = {}

for comp in components:
    ck = frozenset(comp)
    slack = slack_by_comp[ck]
    non_slack = [x for x in comp if x != slack]
    k = len(non_slack)

    comp_lines = [(u, v, Xpu, eid, feat) for (u, v, Xpu, eid, feat) in lines_pu if (u in comp and v in comp)]
    m = len(comp_lines)

    if m == 0 or k == 0:
        PTDF_by_comp[ck] = {
            "PTDF": np.zeros((m, k)),
            "line_ids": [x[3] for x in comp_lines],
            "pos": {},
            "lines": comp_lines,
            "slack": slack,
            "non_slack": non_slack,
        }
        continue

    # B_rr
    comp_idx = [node_index[x] for x in comp]
    B_sub = B[np.ix_(comp_idx, comp_idx)]

    comp_to_local = {node_index[x]: i for i, x in enumerate(comp)}
    mask_local = [comp_to_local[node_index[x]] for x in non_slack]
    B_rr = B_sub[np.ix_(mask_local, mask_local)]

    # A_r
    pos = {node: i for i, node in enumerate(non_slack)}
    A_r = np.zeros((m, k), dtype=float)
    line_ids = []

    for ell, (u, v, Xpu, eid, feat) in enumerate(comp_lines):
        line_ids.append(eid)
        if u != slack:
            A_r[ell, pos[u]] = +1.0
        if v != slack:
            A_r[ell, pos[v]] = -1.0

    B_ell = np.diag([1.0 / Xpu for (_, _, Xpu, _, _) in comp_lines])

    # PTDF = B_ell * A_r * B_rr^{-1}  (wie in code: solve + transpose)
    M = np.linalg.solve(B_rr, A_r.T)   # (k x m)
    PTDFm = B_ell @ M.T                # (m x k)

    PTDF_by_comp[ck] = {"PTDF": PTDFm, "line_ids": line_ids, "pos": pos, "lines": comp_lines, "slack": slack, "non_slack": non_slack}

print("PTDF built for", len(PTDF_by_comp), "components")


PTDF built for 1 components


### Test 1: PTDF-Flows für Injektion am BESS

In [18]:
BESS_ID = "BESS01"
P_INJ_MW = float(node_features.get(BESS_ID, {}).get("p_max_MW", 48.0))

# Komponente mit BESS finden
bess_comp = None
for comp in components:
    if BESS_ID in comp:
        bess_comp = comp
        break
assert bess_comp is not None, "Keine Komponente für BESS gefunden."

ck = frozenset(bess_comp)
PTDFm   = PTDF_by_comp[ck]["PTDF"]
line_ids = PTDF_by_comp[ck]["line_ids"]
pos     = PTDF_by_comp[ck]["pos"]
slack   = PTDF_by_comp[ck]["slack"]

if BESS_ID == slack or BESS_ID not in pos:
    raise RuntimeError(f"BESS {BESS_ID} ist Slack oder nicht in non_slack -> keine PTDF-Spalte möglich.")

# Injektion nur am BESS (MW)
p_r = np.zeros(len(pos))
p_r[pos[BESS_ID]] = P_INJ_MW

flow_ptdf = PTDFm @ p_r  # MW

# Limits @ UTIL
P_limit_eff = {}
for eid in line_ids:
    limA = edge_meta[eid]["limit_A"]
    if limA in (None, ""):
        P_limit_eff[eid] = np.nan
    else:
        P_limit_eff[eid] = amp_to_mw_limit(float(limA), V_KV_DEFAULT, COSPHI_MIN) * util_scale

df_ptdf = pd.DataFrame({
    "edge_id": line_ids,
    "flow_ptdf_MW": flow_ptdf,
    "P_limit_eff_MW": [P_limit_eff[eid] for eid in line_ids],
})
df_ptdf["util"] = np.abs(df_ptdf["flow_ptdf_MW"]) / df_ptdf["P_limit_eff_MW"]
df_ptdf.sort_values("util", ascending=False)


,edge_id,flow_ptdf_MW,P_limit_eff_MW,util
2,E03,-48.0,36.199862,1.325972
0,E01,-48.0,81.449689,0.589321
1,E02,0.0,54.299793,0.000000


### Test 2: DC-Loadflow Solve + Finite Difference vs. PTDF

-> Validiert die Berechnungslogik der PTDF gegen DC-Loadflow.

In [19]:
def dc_solve_flows_for_component(comp, slack, P_inj_by_node):
    """
    Minimaler DC-Loadflow für eine Komponente:
    - Slack-Korrektur
    - theta solve mit B_rr
    - line flows
    """
    # P (MW) + Slack-Korrektur
    P = {n: float(P_inj_by_node.get(n, 0.0)) for n in comp}
    mismatch = float(sum(P.values()))
    P[slack] = P.get(slack, 0.0) - mismatch

    non_slack = [x for x in comp if x != slack]
    if len(non_slack) == 0:
        return {}

    # B_rr
    comp_idx = [node_index[x] for x in comp]
    B_sub = B[np.ix_(comp_idx, comp_idx)]

    comp_to_local = {node_index[x]: i for i, x in enumerate(comp)}
    mask_local = [comp_to_local[node_index[x]] for x in non_slack]
    B_rr = B_sub[np.ix_(mask_local, mask_local)]

    # solve theta (non-slack)
    P_ns = np.array([P[n] for n in non_slack], dtype=float)
    theta_ns = np.linalg.solve(B_rr, (P_ns / S_BASE_MVA))

    theta = {slack: 0.0}
    for n, th in zip(non_slack, theta_ns):
        theta[n] = float(th)

    # flows
    flows = {}
    for (u, v, Xpu, eid, feat) in lines_pu:
        if (u in comp) and (v in comp):
            f_pu = (theta[u] - theta[v]) / Xpu
            flows[eid] = float(f_pu * S_BASE_MVA)
    return flows


DELTA_P = 1.0

# Basecase: alles 0
P0 = {n: 0.0 for n in bess_comp}

# Perturbation: +DELTA_P am BESS
P1 = {n: 0.0 for n in bess_comp}
P1[BESS_ID] = DELTA_P

F0_dc = dc_solve_flows_for_component(bess_comp, slack, P0)
F1_dc = dc_solve_flows_for_component(bess_comp, slack, P1)

dF_dc = pd.Series({eid: (F1_dc.get(eid, 0.0) - F0_dc.get(eid, 0.0)) for eid in line_ids}, name="dF_dc")
dF_ptdf = pd.Series(PTDFm[:, pos[BESS_ID]] * DELTA_P, index=line_ids, name="dF_ptdf")

cmp = pd.concat([dF_dc, dF_ptdf], axis=1)
cmp["abs_err"] = (cmp["dF_dc"] - cmp["dF_ptdf"]).abs()
cmp["rel_err"] = cmp["abs_err"] / (cmp["dF_dc"].abs() + 1e-12)

cmp.sort_values("abs_err", ascending=False)


,dF_dc,dF_ptdf,abs_err,rel_err
E02,-2.562275e-16,0.0,2.562275e-16,2.561618e-04
E03,-1.000000e+00,-1.0,2.220446e-16,2.220446e-16
E01,-1.000000e+00,-1.0,0.000000e+00,0.000000e+00


### Ergebnis-Zusammenfassung (was wurde validiert?)

In [20]:
print("max abs err:", float(cmp["abs_err"].max()))
print("max rel err:", float(cmp["rel_err"].max()))

# Optional: Quick sanity check, dass PTDF-Flow bei p_max konsistent ist:
# dF_ptdf (pro MW) * P_INJ_MW sollte grob flow_ptdf liefern (numerische Rundung erlaubt)
approx = pd.Series(PTDFm[:, pos[BESS_ID]] * P_INJ_MW, index=line_ids, name="PTDF*P_INJ")
check = pd.concat([approx, pd.Series(flow_ptdf, index=line_ids, name="flow_ptdf")], axis=1)
check["diff"] = check["PTDF*P_INJ"] - check["flow_ptdf"]
check


max abs err: 2.5622746654442585e-16
max rel err: 0.0002561618308474824


,PTDF*P_INJ,flow_ptdf,diff
E01,-48.0,-48.0,0.0
E02,0.0,0.0,0.0
E03,-48.0,-48.0,0.0
